# `agent_pools` 导入排查 + OpenRouter API Key 有效性验证

这个 Notebook 用于两件事：
1. 排查 `ModuleNotFoundError: No module named 'agent_pools'`
2. 验证 OpenRouter API Key 是否有效（最小请求）

按顺序逐格运行即可。

## 1) 检查运行环境与模块搜索路径
输出解释器、工作目录和 `sys.path`，确认 Notebook 内核是否与你命令行一致。

In [1]:
import os
import sys
from pathlib import Path

print('Python executable:', sys.executable)
print('cwd:', os.getcwd())
print('\nTop 10 sys.path:')
for index, item in enumerate(sys.path[:10]):
    print(f'[{index}] {item}')

Python executable: /opt/miniconda3/envs/wxj/bin/python
cwd: /Users/wangxingjia/code/ai_homework/lianghua/agent_pools/alpha_agent_demo

Top 10 sys.path:
[0] /opt/miniconda3/envs/wxj/lib/python314.zip
[1] /opt/miniconda3/envs/wxj/lib/python3.14
[2] /opt/miniconda3/envs/wxj/lib/python3.14/lib-dynload
[3] 
[4] /opt/miniconda3/envs/wxj/lib/python3.14/site-packages


## 2) 在 Notebook 中定位项目根目录
逐级向上查找包含 `agent_pools/` 的根目录，并打印关键结构。

In [2]:
import os
from pathlib import Path

start = Path.cwd().resolve()
project_root = None

for candidate in [start, *start.parents]:
    if (candidate / 'agent_pools').exists():
        project_root = candidate
        break

print('Detected project_root:', project_root)
if project_root is None:
    raise RuntimeError('未找到包含 agent_pools/ 的项目根目录，请手动切换 cwd 后重试。')

print('\nproject_root 关键文件:')
for name in ['agent_pools', 'data', 'README.md', 'pyproject.toml']:
    path = project_root / name
    print(f'- {name}:', 'exists' if path.exists() else 'missing')

Detected project_root: /Users/wangxingjia/code/ai_homework/lianghua

project_root 关键文件:
- agent_pools: exists
- data: exists
- README.md: exists
- pyproject.toml: exists


## 3) 临时修复 `agent_pools` 导入问题
将项目根目录追加到 `sys.path`，验证 `from agent_pools.openrouter_config import setup_openrouter_env`。

In [3]:
import sys
import traceback

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print('sys.path[0]:', sys.path[0])

try:
    from agent_pools.openrouter_config import setup_openrouter_env
    print('✅ 导入成功: setup_openrouter_env')
except Exception as exc:
    print('❌ 导入失败:', repr(exc))
    traceback.print_exc()

sys.path[0]: /Users/wangxingjia/code/ai_homework/lianghua
✅ 导入成功: setup_openrouter_env


## 4) 配置并读取 OpenRouter API Key
读取环境变量并调用 `setup_openrouter_env()`，仅显示掩码后的密钥。

In [4]:
import os

setup_openrouter_env()

openrouter_key = os.getenv('OPENROUTER_API_KEY', '')
openrouter_base = os.getenv('OPENROUTER_BASE_URL', 'https://openrouter.ai/api/v1')
openrouter_model = os.getenv('OPENROUTER_MODEL', 'openai/gpt-4o-mini')

mapped_openai_key = os.getenv('OPENAI_API_KEY', '')
mapped_openai_base = os.getenv('OPENAI_BASE_URL', '')


def mask_key(key: str) -> str:
    if not key:
        return '(empty)'
    if len(key) <= 10:
        return key[:3] + '***'
    return key[:7] + '...' + key[-4:]

print('OPENROUTER_API_KEY:', mask_key(openrouter_key))
print('OPENROUTER_BASE_URL:', openrouter_base)
print('OPENROUTER_MODEL:', openrouter_model)
print('Mapped OPENAI_API_KEY:', mask_key(mapped_openai_key))
print('Mapped OPENAI_BASE_URL:', mapped_openai_base)

if not openrouter_key and not mapped_openai_key:
    raise RuntimeError('未检测到 OPENROUTER_API_KEY/OPENAI_API_KEY，请先在环境变量中设置。')

OPENROUTER_API_KEY: os.gete...EY")
OPENROUTER_BASE_URL: https://openrouter.ai/api/v1
OPENROUTER_MODEL: deepseek/deepseek-chat
Mapped OPENAI_API_KEY: os.gete...EY")
Mapped OPENAI_BASE_URL: https://openrouter.ai/api/v1


## 5) 发送最小化 API 请求验证 Key 有效性
向 OpenRouter 聊天接口发送最小请求，打印状态码和响应片段。

In [ ]:
import requests
import json
OPENROUTER_API_KEY
api_key = os.getenv('OPENROUTER_API_KEY') or os.getenv('OPENAI_API_KEY')
base_url = os.getenv('OPENROUTER_BASE_URL') or os.getenv('OPENAI_BASE_URL') or 'https://openrouter.ai/api/v1'
model = os.getenv('OPENROUTER_MODEL') or os.getenv('OPENAI_MODEL') or 'openai/gpt-4o-mini'

url = base_url.rstrip('/') + '/chat/completions'
headers = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json',
}

payload = {
    'model': model,
    'messages': [
        {'role': 'user', 'content': 'Reply with exactly: ok'}
    ],
    'max_tokens': 8,
    'temperature': 0
}

status_code = None
response_json = None
response_text = None

try:
    resp = requests.post(url, headers=headers, json=payload, timeout=20)
    status_code = resp.status_code
    try:
        response_json = resp.json()
    except Exception:
        response_text = resp.text[:500]

    print('Request URL:', url)
    print('Model:', model)
    print('Status code:', status_code)

    if response_json is not None:
        print('Response JSON snippet:')
        print(json.dumps(response_json, ensure_ascii=False, indent=2)[:800])
    else:
        print('Response text snippet:')
        print(response_text)

except requests.Timeout:
    print('请求超时：请检查网络连通性或代理设置。')
except requests.RequestException as exc:
    print('网络请求异常:', repr(exc))

/opt/miniconda3/envs/wxj/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Request URL: https://openrouter.ai/api/v1/chat/completions
Model: deepseek/deepseek-chat
Status code: 401
Response JSON snippet:
{
  "error": {
    "message": "Missing Authentication header",
    "code": 401
  }
}


## 6) 根据响应状态码输出诊断结果
按 `200/401/403/429/5xx` 给出结论和下一步排查建议。

In [ ]:
if status_code is None:
    print('❌ 未拿到状态码（请求可能未发出）。')
elif status_code == 200:
    print('✅ 诊断结果：API Key 有效，请求成功。')
elif status_code == 401:
    print('❌ 诊断结果：API Key 无效或已过期（401 Unauthorized）。')
    print('建议：确认 OPENROUTER_API_KEY 是否最新、是否多了空格、是否对应正确账户。')
elif status_code == 403:
    print('⚠️ 诊断结果：权限不足（403 Forbidden）。')
    print('建议：检查模型访问权限、账户策略、组织权限设置。')
elif status_code == 429:
    print('⚠️ 诊断结果：触发限流或额度限制（429 Too Many Requests）。')
    print('建议：检查余额/额度、降低并发、重试并增加退避。')
elif 500 <= status_code <= 599:
    print(f'⚠️ 诊断结果：服务端异常（{status_code}）。')
    print('建议：稍后重试，并检查 OpenRouter 状态页。')
else:
    print(f'⚠️ 诊断结果：未归类状态码 {status_code}。')
    print('建议：检查请求头（Authorization）、模型名、网络代理和超时设置。')

print('\n下一步排查建议：')
print('- 确认模型名是否可用（OPENROUTER_MODEL）。')
print('- 检查是否设置了代理/公司网络拦截。')
print('- 如在 VS Code Notebook，确认内核与终端是同一 Python 环境。')

### 额外：检查 `POE_API_KEY`
如果你希望用 Poe 的 key 来做测试，请在运行下面的代码前，在终端或 Notebook 环境中设置 `POE_API_KEY` 环境变量。
运行第 4) 和 5) 步可以把（经 `setup_openrouter_env()` 映射后的）`OPENAI_API_KEY` 用于请求验证。

In [1]:
import os

def mask_key(key: str) -> str:
    if not key:
        return '(empty)'
    if len(key) <= 10:
        return key[:3] + '***'
    return key[:7] + '...' + key[-4:]

print('POE_API_KEY:', mask_key(os.getenv('POE_API_KEY', '')))
print('Mapped OPENAI_API_KEY (before running setup_openrouter_env()):', mask_key(os.getenv('OPENAI_API_KEY', '')))

print('说明：')
print('- 若你已在 Notebook 环境中设置了 POE_API_KEY，请先运行第 4) 的 cell（会调用 setup_openrouter_env()）以完成映射，然后运行第 5) 的请求验证。')
print('- 我这里无法替你执行带有你私钥的网络请求；请在本地运行第 5) cell 来完成验证。')

POE_API_KEY: (empty)
Mapped OPENAI_API_KEY (before running setup_openrouter_env()): (empty)
说明：
- 若你已在 Notebook 环境中设置了 POE_API_KEY，请先运行第 4) 的 cell（会调用 setup_openrouter_env()）以完成映射，然后运行第 5) 的请求验证。
- 我这里无法替你执行带有你私钥的网络请求；请在本地运行第 5) cell 来完成验证。


In [ ]:
import os
import subprocess
import json

def mask_key(key: str) -> str:
    if not key:
        return '(empty)'
    if len(key) <= 10:
        return key[:3] + '***'
    return key[:7] + '...' + key[-4:]

print('os.getenv(POE_API_KEY):', mask_key(os.getenv('POE_API_KEY', '')))
print('os.environ.get(OPENAI_API_KEY):', mask_key(os.environ.get('OPENAI_API_KEY', '')))

# 尝试通过一个交互式 zsh 登录 shell 读取（~/.zshrc 通常只对登录 shell 生效）
try:
    zsh_out = subprocess.run(['zsh','-i','-c','echo $POE_API_KEY'], capture_output=True, text=True, timeout=5).stdout.strip()
    print('zsh -i -c echo $POE_API_KEY:', mask_key(zsh_out))
except Exception as e:
    print('Failed to run zsh check:', repr(e))

# 通过 setup_openrouter_env() 做映射（如果可用）
try:
    from agent_pools.openrouter_config import setup_openrouter_env
    print('Found setup_openrouter_env(), running it...')
    setup_openrouter_env()
    print('After mapping, OPENAI_API_KEY:', mask_key(os.getenv('OPENAI_API_KEY','')))
except Exception as e:
    print('Could not import/run setup_openrouter_env():', repr(e))

# 可选：发送最小化请求验证密钥（需要 requests 可用并联网）
proceed = input("是否发送最小化请求验证密钥？输入 y 继续，否则回车跳过：")
if proceed.lower().startswith('y'):
    api_key = os.getenv('OPENROUTER_API_KEY') or os.getenv('OPENAI_API_KEY') or os.getenv('POE_API_KEY')
    if not api_key:
        print('未检测到可用的 API Key，退出。')
    else:
        base_url = os.getenv('OPENROUTER_BASE_URL') or os.getenv('OPENAI_BASE_URL') or 'https://openrouter.ai/api/v1'
        model = os.getenv('OPENROUTER_MODEL') or os.getenv('OPENAI_MODEL') or 'openai/gpt-4o-mini'
        url = base_url.rstrip('/') + '/chat/completions'
        headers = {'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'}
        payload = {'model': model, 'messages': [{'role': 'user', 'content': 'Reply with exactly: ok'}], 'max_tokens': 8, 'temperature': 0}
        try:
            import requests
            resp = requests.post(url, headers=headers, json=payload, timeout=20)
            print('Status code:', resp.status_code)
            try:
                print('Response snippet:', json.dumps(resp.json(), ensure_ascii=False)[:800])
            except Exception:
                print('Response text snippet:', resp.text[:500])
        except Exception as e:
            print('请求异常：', repr(e))
else:
    print('跳过发送请求。')